# 02 — base data → ESM-C 600M(토큰별) → ProteomeLM-L(328M) layer 8

`01-build_basedata.ipynb`가 만든 균주별 TSV(`data/basedata/*.tsv`, 순번대로 이어 붙인다)의 `seq_aa`를 두 단계로 임베딩한다.

| 단계 | 산출물 | 내용 |
|---|---|---|
| Stage A | `data/emb_esm_600m.h5` | 단백질별 ESM-C 600M **토큰별** 임베딩 (BOS/EOS 포함, `is_special`로 표시). 파일이 있으면 `BUILD_ESM` 가드로 건너뛴다 |
| Stage B | `data/emb_esm_600m_mean.h5` | 저장된 전 토큰 평균 `[n_prot, 1152]` |
| Stage B | `data/emb_proteome_328m.h5` | 위 벡터를 프로테옴 단위로 ProteomeLM-L에 통과시킨 `hidden_states[PLM_LAYER]` |

h5는 **임베딩 + 행 정합 키(`org_id`·`gene_id`)** 만 담는다. 라벨·메타·서열은 항상 TSV에서 참조하며, h5의 $i$ 번 행 = TSV의 $i$ 번 행이다. 프로테옴 경계는 `org_id`가 바뀌는 위치에서 유도한다.

In [ ]:
!pip install esm transformers h5py numpy tqdm hf_transfer typing_extensions
# !pip install git+https://github.com/Bitbol-Lab/ProteomeLM.git
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install --upgrade typing_extensions

  Using cached h5py-3.16.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached hf_transfer-0.1.9-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.7 kB)
  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached biotite-1.7.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
  Using cached msgpack_numpy-0.4.8-py2.py3-none-any.whl.metadata (5.0 kB)
  Using cached biopython-1.88-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached scikit_learn-1.9.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.3 kB)
  Using cached brotli-1.2.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.1 kB)
  Using cached pandas-3.0.6-cp

In [2]:
import os, h5py, torch, numpy as np
from esm.models.esmc import ESMC
from proteomelm import ProteomeLMForMaskedLM
from tqdm import tqdm
from proteomefit.data import load_basedata

BASE_DIR='../data/basedata'           # 01 산출물 (균주별 TSV) - 라벨·메타·서열의 출처
ESM_PATH='../data/emb_esm_600m.h5'; OUT_PATH='../data/emb_proteome_328m.h5'
MEAN_PATH='../data/emb_esm_600m_mean.h5'  # Stage B 부산물 - 유전자당 ESM 토큰 평균 [n_prot,1152]
ESM_MODEL='esmc_600m'; PLM_MODEL='Bitbol-Lab/ProteomeLM-L'; DEVICE='cuda:0'
PLM_LAYER=8                            # hidden_states[8] = 8번째 블록 출력
MAX_LEN=4096                           # 잔기 상한 (ProteomeLM 입력 절단 길이)
BATCH_SIZE=32; MAX_TOKENS=16000
ID_COLS=['org_id','gene_id']           # h5에 저장할 행 정합 키

In [3]:
# TSV에서 3열만 적재 - 이 행 순서가 그대로 임베딩 h5의 행 순서
assert os.path.isdir(BASE_DIR), f'{BASE_DIR} not found -> run 01-build_basedata.ipynb first'
base=load_basedata(BASE_DIR,usecols=ID_COLS+['seq_aa'],dtype=str,keep_default_na=False)   # 균주별 TSV 를 순번대로 이어 붙인다
seqs=[s[:MAX_LEN] for s in base.seq_aa.tolist()]   # 앞 MAX_LEN 잔기로 절단
trunc=(base.seq_aa.str.len().to_numpy()>MAX_LEN).astype(np.int8)   # 1=절단됨 [n_prot]

# 균주 블록 경계 - org_id가 바뀌는 위치 [n_org]
oid=base.org_id.to_numpy(); bnd=np.flatnonzero(np.r_[True,oid[1:]!=oid[:-1],True])
org_start,org_end=bnd[:-1],bnd[1:]
assert len(org_start)==base.org_id.nunique(), 'org_id rows are not contiguous per organism'

def write_ids(h5):                     # 행 정합 키와 attrs 기록
    vlen=h5py.string_dtype('utf-8')
    for c in ID_COLS: h5.create_dataset(c,data=base[c].to_numpy(),dtype=vlen)
    h5.attrs['base_data']=BASE_DIR; h5.attrs['n_proteins']=len(base)

BUILD_ESM=not os.path.exists(ESM_PATH)
if not BUILD_ESM:                      # 기존 파일의 단백질 수가 현재 base data와 같은지 확인
    with h5py.File(ESM_PATH,'r') as h5:
        assert int(h5.attrs['n_proteins'])==len(base), \
            f'{ESM_PATH}: n_proteins {int(h5.attrs["n_proteins"]):,} != base {len(base):,} -> delete it and rebuild'
print(f'proteins:{len(base):,} | organisms:{len(org_start)} | residues:{sum(map(len,seqs)):,} '
      f'| truncated at MAX_LEN={MAX_LEN}: {int(trunc.sum()):,}')
print('Stage A (ESM):','BUILD' if BUILD_ESM else f'SKIP (found {ESM_PATH})')

proteins:217,468 | organisms:46 | residues:71,405,495 | truncated at MAX_LEN=4096: 80
Stage A (ESM): BUILD


## Stage A — ESM-C 600M 토큰별 임베딩 → `data/emb_esm_600m.h5`

레이아웃: `emb`[총토큰, ESM_DIM] · `is_special`(1=특수토큰) · `offsets`[n_prot+1] · 정합 키. 배치 pad만 제외하고 특수토큰은 포함해 저장한다.

길이 상한은 **`MAX_LEN=4096` 잔기** 이며, 절단 여부는 `is_truncated` 데이터셋과 `max_len` 속성에 기록한다.

In [4]:
if BUILD_ESM:
    n=len(seqs); res_lens=np.fromiter((len(s) for s in seqs),np.int64,n)   # 잔기 수 [n_prot]

    esm=ESMC.from_pretrained(ESM_MODEL).to(DEVICE).eval()
    SPECIAL=set(getattr(esm.tokenizer,'all_special_ids',[])) or {esm.tokenizer.pad_token_id,esm.tokenizer.cls_token_id,esm.tokenizer.eos_token_id}
    PAD_ID=esm.tokenizer.pad_token_id
    def esm_batch(bs):                                    # 배치 forward -> 단백질별 (토큰 임베딩 [T,D], is_special [T])
        tok=esm._tokenize(bs)
        with torch.no_grad(), torch.autocast('cuda',dtype=torch.bfloat16):
            emb=esm(sequence_tokens=tok).embeddings
        nonpad=tok.ne(PAD_ID)                             # 배치 pad만 제외
        special=torch.zeros_like(tok,dtype=torch.bool)
        for sid in SPECIAL: special|=tok.eq(sid)          # 특수토큰 위치 표시
        emb=emb.to(torch.float16).cpu().numpy(); nonpad=nonpad.cpu().numpy(); special=special.cpu().numpy()
        return [(emb[i][nonpad[i]], special[i][nonpad[i]].astype(np.int8)) for i in range(len(bs))]
    def batches(sl):
        out=[]; st=0
        for i in range(len(sl)):
            if i>st and (i-st+1>BATCH_SIZE or (i-st+1)*int(sl[i])>MAX_TOKENS): out.append((st,i)); st=i
        out.append((st,len(sl))); return out
    # 특수토큰 수 프로브 -> 저장 토큰 수 = 잔기 수 + N_SPECIAL
    pe,ps=esm_batch(['MA'])[0]; ESM_DIM=int(pe.shape[-1]); N_SPECIAL=int(ps.sum())
    tok_lens=res_lens+N_SPECIAL
    offsets=np.concatenate([[0],np.cumsum(tok_lens)]).astype(np.int64); order=np.argsort(tok_lens,kind='stable')
    print(f'proteins:{n:,} | tokens(incl special):{int(offsets[-1]):,} | ESM_DIM:{ESM_DIM} | N_SPECIAL:{N_SPECIAL} | ~fp16 {int(offsets[-1])*ESM_DIM*2/1e9:.1f} GB')

    with h5py.File(ESM_PATH,'w') as h5:
        emb_ds=h5.create_dataset('emb',shape=(int(offsets[-1]),ESM_DIM),dtype='float16')
        spec_ds=h5.create_dataset('is_special',shape=(int(offsets[-1]),),dtype='int8')   # 1=특수토큰, 0=잔기
        for s,e in tqdm(batches(tok_lens[order]),desc='ESM'):
            bidx=order[s:e]
            for j,(res,sp) in zip(bidx,esm_batch([seqs[j] for j in bidx])):
                lo,hi=int(offsets[j]),int(offsets[j+1]); assert hi-lo==len(res)
                emb_ds[lo:hi]=res; spec_ds[lo:hi]=sp
        h5.create_dataset('offsets',data=offsets)
        h5.create_dataset('is_truncated',data=trunc)      # 1=MAX_LEN으로 절단된 단백질
        write_ids(h5)
        h5.attrs['model']='ESM C 600M (esmc_600m)'; h5.attrs['embed_dim']=ESM_DIM; h5.attrs['n_special']=N_SPECIAL
        h5.attrs['max_len']=MAX_LEN
        h5.attrs['layout']='per-token incl. special (BOS first, EOS last); is_special=1 marks specials; protein i = emb[offsets[i]:offsets[i+1]] = row i of base_data'
    print('Saved',ESM_PATH)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/608 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

data/weights/esmc_600m_2024_12_v0.pth:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

proteins:217,468 | tokens(incl special):71,840,431 | ESM_DIM:1152 | N_SPECIAL:2 | ~fp16 165.5 GB


ESM: 100%|██████████| 7346/7346 [39:52<00:00,  3.07it/s]


Saved data/emb_esm_600m.h5


## Stage B — 토큰 평균 → `emb_esm_600m_mean.h5` + ProteomeLM-L layer 8 → `emb_proteome_328m.h5`

단백질별로 Stage A가 저장한 전 토큰 평균을 낸 뒤, 그 벡터를 `emb_esm_600m_mean.h5`에 저장하고 `org_start`/`org_end`로 자른 프로테옴 전체를 ProteomeLM-L에 통과시켜 `hidden_states[PLM_LAYER]`를 얻는다. 두 산출물 모두 행 순서가 TSV와 1:1이다.

In [5]:
plm=ProteomeLMForMaskedLM.from_pretrained(PLM_MODEL).to(DEVICE).eval()
PLM_DTYPE=next(plm.parameters()).dtype; IN_DIM=int(plm.config.input_size)

# hidden_states 인덱스 규약 확인 - [0]=입력 임베딩, [k]=k번째 블록 출력
with torch.no_grad():
    probe=plm(inputs_embeds=torch.zeros((1,2,IN_DIM),dtype=PLM_DTYPE,device=DEVICE),
              attention_mask=torch.ones((1,2),dtype=torch.long,device=DEVICE),output_hidden_states=True)
N_HIDDEN=len(probe.hidden_states)
assert 0<PLM_LAYER<N_HIDDEN, f'PLM_LAYER {PLM_LAYER} out of range (n_hidden={N_HIDDEN})'
print(f'hidden_states:{N_HIDDEN} (0=input embedding, 1..{N_HIDDEN-1}=block outputs) | using layer {PLM_LAYER}')

def plm_embed(mean_vecs):                                 # [n_prot,IN_DIM] -> layer PLM_LAYER 은닉표현 [n_prot,EMB_DIM]
    x=torch.from_numpy(mean_vecs).to(DEVICE,PLM_DTYPE).unsqueeze(0)
    attn=torch.ones(x.shape[:2],dtype=torch.long,device=DEVICE)
    with torch.no_grad(): out=plm(inputs_embeds=x,attention_mask=attn,output_hidden_states=True)
    return out.hidden_states[PLM_LAYER][0].to(torch.float16).cpu().numpy()
EMB_DIM=int(plm_embed(np.zeros((2,IN_DIM),np.float32)).shape[-1])

with h5py.File(ESM_PATH,'r') as fin, h5py.File(OUT_PATH,'w') as fout, h5py.File(MEAN_PATH,'w') as fmean:
    offsets=fin['offsets'][:]; emb_in=fin['emb']; ESM_DIM=emb_in.shape[1]
    assert ESM_DIM==IN_DIM, f'ESM {ESM_DIM} != ProteomeLM input {IN_DIM}'
    N=len(offsets)-1; print(f'proteins:{N:,} | ProteomeLM embed_dim:{EMB_DIM} | dtype:{PLM_DTYPE}')
    emb_out=fout.create_dataset('emb',shape=(N,EMB_DIM),dtype='float16')
    mean_out=fmean.create_dataset('emb',shape=(N,ESM_DIM),dtype='float16')   # ProteomeLM 입력과 동일한 벡터
    for s,e in zip(tqdm(org_start,desc='ProteomeLM'),org_end):
        means=np.empty((e-s,ESM_DIM),np.float32)
        for k,i in enumerate(range(s,e)):
            lo,hi=int(offsets[i]),int(offsets[i+1])
            means[k]=emb_in[lo:hi].mean(0)      # 저장된 전 토큰 평균 [ESM_DIM]
        mean_out[s:e]=means.astype(np.float16)  # 유전자 단위 ESM 벡터
        emb_out[s:e]=plm_embed(means)
    for h5 in (fout,fmean): write_ids(h5)
    fout.attrs['proteomelm_model']=PLM_MODEL; fout.attrs['embed_dim']=EMB_DIM
    fout.attrs['layer']=PLM_LAYER; fout.attrs['n_hidden_states']=N_HIDDEN
    fout.attrs['layout']=f'per-protein hidden_states[{PLM_LAYER}]; protein i = emb[i] = row i of base_data; proteome = rows with same org_id'
    fout.attrs['n_organisms']=len(org_start)
    fmean.attrs['model']='ESM C 600M (esmc_600m) token mean'; fmean.attrs['embed_dim']=ESM_DIM
    fmean.attrs['layout']='per-protein token mean incl. BOS/EOS (matches ProteomeLM average_representation); protein i = emb[i] = row i of base_data'
print('Saved',OUT_PATH,f'(dim {EMB_DIM}, layer {PLM_LAYER}) |',MEAN_PATH,'(dim',ESM_DIM,') | proteins:',N)

hidden_states:19 (0=input embedding, 1..18=block outputs) | using layer 8
proteins:217,468 | ProteomeLM embed_dim:1152 | dtype:torch.float32


ProteomeLM: 100%|██████████| 46/46 [08:19<00:00, 10.86s/it]


Saved data/emb_proteome_328m.h5 (dim 1152, layer 8) | data/emb_esm_600m_mean.h5 (dim 1152 ) | proteins: 217468


In [6]:
# 검증 - 두 산출 h5의 행이 TSV 행과 1:1 정합하는지
for path in (OUT_PATH,MEAN_PATH):
    with h5py.File(path,'r') as h5:
        print(path,'| emb:',h5['emb'].shape,h5['emb'].dtype,'| base:',h5.attrs['base_data'])
        ids={c:h5[c][:].astype(str) for c in ID_COLS}
        assert all(np.array_equal(ids[c],base[c].to_numpy()) for c in ID_COLS), f'{path} rows misaligned with TSV'
        print(f'  rows aligned:{len(ids["gene_id"]):,} | row0: {ids["org_id"][0]} {ids["gene_id"][0]}')

data/emb_proteome_328m.h5 | emb: (217468, 1152) float16 | base: data/feba_basedata.tsv
  rows aligned:217,468 | row0: acidovorax_3H11 Ac3H11_3791
data/emb_esm_600m_mean.h5 | emb: (217468, 1152) float16 | base: data/feba_basedata.tsv
  rows aligned:217,468 | row0: acidovorax_3H11 Ac3H11_3791
